# Bambara Speech AI - Google Colab Training Notebook

This notebook is designed to prepare and run training for Bambara speech workloads, including:
- TTS (Text-to-Speech)
- ASR (Automatic Speech Recognition)
- dataset manifest creation
- preprocessing
- inference

Important:
- This is a starter notebook that expects a real Bambara dataset with audio + transcripts.
- For real training, use a GPU-enabled runtime or Colab Pro/Pro+ if available.
- Replace the sample data paths with your own dataset before training.


In [6]:
# 1) Install dependencies
!python -V
!pip install -q --upgrade pip
!pip install -q numpy pandas soundfile librosa scipy matplotlib tqdm PyYAML
!pip install -q torch torchaudio transformers accelerate datasets sentencepiece scikit-learn
!pip install -q git+https://github.com/huggingface/transformers.git
print('Dependencies installed')


Python 3.13.16
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Dependencies installed


In [7]:
# 2) Mount Google Drive (optional but recommended)
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted


In [8]:
# 3) Clone or open project repository
import os
repo_path = '/content/bambara-tts-waxal-clean'
if not os.path.exists(repo_path):
    !git clone https://github.com/adamacoulibalyad670-droid/bambara-tts-waxal-clean.git {repo_path}
else:
    print('Repository already exists')

%cd {repo_path}
!ls -la


Cloning into '/content/bambara-tts-waxal-clean'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 49 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 13.33 KiB | 758.00 KiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/bambara-tts-waxal-clean
total 52
drwxr-xr-x 7 root root 4096 Oct 26 02:57 .
drwxr-xr-x 1 root root 4096 Oct 26 02:57 ..
-rw-r--r-- 1 root root 6285 Oct 26 02:57 colab_bambara_speech_training.ipynb
-rw-r--r-- 1 root root  401 Oct 26 02:57 COLAB_README.md
drwxr-xr-x 2 root root 4096 Oct 26 02:57 configs
drwxr-xr-x 2 root root 4096 Oct 26 02:57 data
drwxr-xr-x 8 root root 4096 Oct 26 02:57 .git
-rw-r--r-- 1 root root  296 Oct 26 02:57 .gitignore
-rw-r--r-- 1 root root 3410 Oct 26 02:57 README.md
-rw-r--r-- 1 root root  268 Oct 26 02:57 requirements.txt
drwxr-xr-x 2 root root 4096 Oct 26 02:57 scripts
drwxr-xr-x 2 root root 409

In [9]:
# 4) Create required folders for dataset
!mkdir -p data/audio/train data/audio/val data/audio/test data/transcripts
!mkdir -p data/processed models/tts models/asr outputs
!find data -maxdepth 3 -type d -print


data
data/transcripts
data/processed
data/audio
data/audio/val
data/audio/train
data/audio/test


## Upload your dataset

You should place your Bambara audio files in the folders above and create transcript files like this:

```text
data/transcripts/train.txt
train/001.wav Aw ka se ka baara duman min ye?
train/002.wav Mali la denw ka caya.
```

For validation and testing, create similar files in `data/transcripts/val.txt` and `data/transcripts/test.txt`.

You can upload files using the Colab Files panel or by copying from Google Drive.


In [10]:
# 5) Build manifest CSV files if transcripts are available
!python scripts/build_manifest.py --audio_dir data/audio/train --transcript_file data/transcripts/train.txt --output data/train_manifest.csv
!python scripts/build_manifest.py --audio_dir data/audio/val --transcript_file data/transcripts/val.txt --output data/val_manifest.csv
!python scripts/build_manifest.py --audio_dir data/audio/test --transcript_file data/transcripts/test.txt --output data/test_manifest.csv
!ls -l data
!head -n 5 data/train_manifest.csv


Traceback (most recent call last):
  File "/content/bambara-tts-waxal-clean/scripts/build_manifest.py", line 63, in <module>
    main()
    ~~~~^^
  File "/content/bambara-tts-waxal-clean/scripts/build_manifest.py", line 24, in main
    raise FileNotFoundError(f"Transcript file not found: {transcript_file}")
FileNotFoundError: Transcript file not found: data/transcripts/train.txt
Traceback (most recent call last):
  File "/content/bambara-tts-waxal-clean/scripts/build_manifest.py", line 63, in <module>
    main()
    ~~~~^^
  File "/content/bambara-tts-waxal-clean/scripts/build_manifest.py", line 24, in main
    raise FileNotFoundError(f"Transcript file not found: {transcript_file}")
FileNotFoundError: Transcript file not found: data/transcripts/val.txt
Traceback (most recent call last):
  File "/content/bambara-tts-waxal-clean/scripts/build_manifest.py", line 63, in <module>
    main()
    ~~~~^^
  File "/content/bambara-tts-waxal-clean/scripts/build_manifest.py", line 24, in main
   

In [11]:
# 6) Preprocess the dataset
!python -m src.preprocess --manifest data/train_manifest.csv --output_dir data/processed
!ls -l data/processed


Processed manifest saved to: data/processed/manifest_processed.csv
Rows kept: 2
total 4
-rw-r--r-- 1 root root 168 Oct 26 02:57 manifest_processed.csv


## Training for TTS (Text-to-Speech)

Choose one of the recommended backends:
- XTTS-v2
- Coqui TTS
- VITS
- FastSpeech2

This notebook uses the repository starter scripts, which are ready to be replaced with a real training backend when your data is prepared.


## Training for ASR (Speech-to-Text)

Choose one of the recommended backends:
- Whisper fine-tuning
- Wav2Vec2
- MMS
- HuBERT

This notebook is ready to plug in a real ASR backbone once your Bambara audio and transcripts are available.


In [12]:
!pip install -q transformers datasets accelerate torch torchaudio pandas
!python /tmp/train.py

python3: can't open file '/tmp/train.py': [Errno 2] No such file or directory


In [13]:
%cd /content/bambara-tts-waxal-clean

import glob, os, torch, soundfile as sf, torchaudio
from pathlib import Path
from speechbrain.pretrained import EncoderClassifier
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan

# ابحث في كل مكان
all_m4a = glob.glob("/content/**/*.m4a", recursive=True)
if not all_m4a:
    all_m4a = glob.glob("/content/**/*.wav", recursive=True)

    print(f"Found {len(all_m4a)} files")
    print(f"Using: {all_m4a[0]}")

    ref_file = all_m4a[0]

    sig, fs = torchaudio.load(ref_file)
    if sig.shape[0]>1: sig = sig.mean(dim=0, keepdim=True)
    sig = torchaudio.functional.resample(sig, fs, 16000)

    spk_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="/tmp/spkrec")
    emb = spk_model.encode_batch(sig).squeeze()

    tmp = torch.zeros(512)
    tmp[:192]=emb; tmp[192:384]=emb; tmp[384:]=emb[:128]
    spk_emb = tmp.unsqueeze(0)

    processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
    model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
    vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

    inputs = processor(text="I ni ce, ne togo ye Waxal", return_tensors="pt")
    spec = model.generate_speech(inputs["input_ids"], spk_emb)
    speech = vocoder(spec)

    speech_np = speech.detach().cpu().numpy().squeeze()
    Path("outputs").mkdir(exist_ok=True, parents=True)
    sf.write("outputs/bambara_WAX_REAL.wav", speech_np, 16000)
    print("✅ DONE: outputs/bambara_WAX_REAL.wav - هذه نبرة Wax الحقيقية!")

/content/bambara-tts-waxal-clean


ModuleNotFoundError: No module named 'speechbrain'

In [ ]:
%cd /content
!rm -rf bambara-tts-waxal-clean
!git clone https://github.com/djiby98/bambara-tts-waxal-clean.git
!ls bambara-tts-waxal-clean/models/tts/bambara_xtts/ | head -20

In [ ]:
%cd /content/bambara-tts-waxal-clean
!find . -name "*.m4a" -type f > /tmp/list.txt
!cat /tmp/list.txt

# الآن اقرأ من الملف الذي وجده find
import torch, soundfile as sf, torchaudio
from pathlib import Path
from speechbrain.pretrained import EncoderClassifier
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan

with open("/tmp/list.txt") as f:
    ref_file = f.readline().strip()

    print(f"Using: {ref_file}")

    sig, fs = torchaudio.load(ref_file)
    if sig.shape[0]>1: sig = sig.mean(dim=0, keepdim=True)
    sig = torchaudio.functional.resample(sig, fs, 16000)

    spk_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="/tmp/spkrec")
    emb = spk_model.encode_batch(sig).squeeze()
    tmp = torch.zeros(512)
    tmp[:192]=emb; tmp[192:384]=emb; tmp[384:]=emb[:128]
    spk_emb = tmp.unsqueeze(0)

    processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
    model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
    vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

    inputs = processor(text="I ni ce, ne togo ye Waxal", return_tensors="pt")
    spec = model.generate_speech(inputs["input_ids"], spk_emb)
    speech = vocoder(spec)
    speech_np = speech.detach().cpu().numpy().squeeze()
    Path("outputs").mkdir(exist_ok=True)
    sf.write("outputs/bambara_WAX_REAL.wav", speech_np, 16000)
    print("✅ DONE - شغل الملف من outputs/bambara_WAX_REAL.wav")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/bambara-tts-waxal-clean
import glob, torch, torchaudio, librosa, soundfile as sf
from pathlib import Path
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from speechbrain.pretrained import EncoderClassifier

ref = glob.glob("/content/drive/MyDrive/**/*.m4a", recursive=True)[0]
print(ref)

sig, fs = torchaudio.load(ref)
sig = sig.mean(dim=0, keepdim=True) if sig.shape[0]>1 else sig
sig_16k = torchaudio.functional.resample(sig, fs, 16000)

spk_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="/tmp/spkrec")
emb = spk_model.encode_batch(sig_16k).squeeze()
tmp = torch.zeros(512)
tmp[:192]=emb
tmp[192:384]=emb
tmp[384:]=emb[:128]
spk_emb = tmp.unsqueeze(0)

processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

inputs = processor(text="I ni ce, ne togo ye Waxal", return_tensors="pt")
spec = model.generate_speech(inputs["input_ids"], spk_emb)
speech = vocoder(spec)
speech_np = speech.detach().cpu().numpy().squeeze()
speech_slow = librosa.effects.time_stretch(speech_np, rate=0.95)

Path("outputs").mkdir(exist_ok=True)
sf.write("outputs/bambara_WAX_FINAL.wav", speech_slow, 16000)
print("DONE FINAL")

In [ ]:
!python -c "import os,urllib.request; os.chdir('/content'); os.system('git clone https://github.com/djiby98/bambara-tts-waxal-clean.git 2>/dev/null'); urllib.request.urlretrieve('https://raw.githubusercontent.com/coqui-ai/TTS/dev/TTS/tts/configs/xtts_config.py','/tmp/x.py'); print('ok')"

In [ ]:
from google.colab import files
files.download('outputs/bambara_WAX_FINAL.wav')

In [ ]:
sf.write("outputs/bambara_WAX_FINAL.wav", speech_slow, 16000)

In [ ]:
import librosa
speech_slow = librosa.effects.pitch_shift(speech_np, sr=16000, n_steps=-1)
speech_slow = librosa.effects.time_stretch(speech_slow, rate=0.92)
sf.write("outputs/bambara_WAX_FINAL.wav", speech_slow, 16000)
from google.colab import files
files.download('outputs/bambara_WAX_FINAL.wav')

In [ ]:
!curl -sL https://raw.githubusercontent.com/djiby98/bambara-tts-waxal-clean/main/run.py -o /tmp/final.py 2>/dev/null; python /tmp/final.py || echo "use local"

In [ ]:
!ls -lh outputs/

In [ ]:
from google.colab import files
files.download("outputs/bambara_WAX_FINAL.wav")

In [ ]:
text="I ni ce, ne togo ye Waxal"

In [ ]:
!pip install -q transformers accelerate

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch, soundfile as sf
from google.colab import files

model_id = "facebook/mms-tts-bam"
print("جاري التحميل...")
model = VitsModel.from_pretrained(model_id)
tok = AutoTokenizer.from_pretrained(model_id)

text = "I ni ce, ne togo ye Waxal"
inputs = tok(text, return_tensors="pt")

with torch.no_grad():
    wave = model(**inputs).waveform[0].cpu().float().numpy()

    sf.write("bambara_native.wav", wave, 16000)
    print("✅ تم bambara_native.wav - حجمه:", len(wave))
    files.download("bambara_native.wav")

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch, soundfile as sf
from google.colab import files

model_id = "facebook/mms-tts-bam"
print("جاري التحميل...")
model = VitsModel.from_pretrained(model_id)
tok = AutoTokenizer.from_pretrained(model_id)

text = "I ni ce, ne togo ye Waxal"
inputs = tok(text, return_tensors="pt")

with torch.no_grad():
    wave = model(**inputs).waveform[0].cpu().float().numpy()

    sf.write("bambara_native.wav", wave, 16000)
    print("✅ تم bambara_native.wav - حجمه:", len(wave))
    files.download("bambara_native.wav")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech

# تأكد الموديل موجود
model_path = Path("models/tts/bambara_xtts").resolve()
print(f"Checking: {model_path}")
print(f"Exists: {model_path.exists()}")
if model_path.exists():
    print("Files:", os.listdir(model_path)[:10])

    # تحميل من المجلد المحلي
    processor = SpeechT5Processor.from_pretrained(model_path, local_files_only=True)
    model = SpeechT5ForTextToSpeech.from_pretrained(model_path, local_files_only=True)
    print("Bambara TTS ready! - I ni ce")

In [ ]:
import os
!find /content -type d -name "*bambara*" 2>/dev/null
!ls -la /content/models/tts/ 2>/dev/null || echo "مجلد tts ما كاينش"
!ls -la /content/drive/MyDrive/ 2>/dev/null | head -20

In [ ]:
!pip install -q speechbrain torchaudio

In [ ]:
!python -m src.infer_tts --text "I ni ce, ne togo ye Waxal"

In [ ]:
%cd /content/bambara-tts-waxal-clean
!python -m src.infer_tts --text "I ni ce, ne togo ye Waxal"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q TTS

In [ ]:
# 8) Train ASR
!python -m src.train_asr --config configs/asr_config.yaml --manifest data/train_manifest.csv


In [ ]:
# 9) Inference examples
!python -m src.infer_tts --text "Aw ka se ka baara duman min ye?"
!python -m src.infer_asr --audio_path data/audio/test/001.wav


## Notes

This notebook is a starter implementation for Bambara speech AI. It prepares the workflow and project structure, but real model performance depends on:
- dataset quality
- transcript correctness
- adequate GPU memory
- careful preprocessing and validation
- use of a production-grade TTS/ASR backbone


In [ ]:
!pip install -q datasets soundfile && python -c "from datasets import load_dataset; import os, soundfile as sf; ds=load_dataset('adama-coulibalyad670/bambara-tts-waxal-clean'); print(ds); os.makedirs('data/audio/train', exist_ok=True); os.makedirs('data/transcripts', exist_ok=True); open('data/transcripts/train.txt','w',encoding='utf-8').close(); f=open('data/transcripts/train.txt','w',encoding='utf-8'); [ (sf.write(f'data/audio/train/{i:05d}.wav', s['audio']['array'], s['audio']['sampling_rate']), f.write(f\"{i:05d}.wav|{s.get('text') or s.get('transcription') or s.get('sentence') or s.get('transcript')}\n\")) for i,s in enumerate(ds['train']) ]; f.close(); print('Done!');"

In [ ]:
!echo "text,audio_path,speaker_id,language" > data/train_manifest.csv!wc -l data/transcripts/train.txt
!ls data/audio/train | wc -l

In [ ]:
!echo "text,audio_path,speaker_id,language" > data/train_manifest.csv

In [ ]:
!awk -F'|' '{gsub(/"/,""); printf "\"%s\",data/audio/train/%s,speaker_01,bm\n", $2, $1}' data/transcripts/train.txt >> data/train_manifest.csv

In [ ]:
!wc -l data/train_manifest.csv && head -n 3 data/train_manifest.csv